In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = Path(r"C:\Users\polok\deep-learning-assignment-1")
DATA_DIR = PROJECT_ROOT / "data"

print("Project exists:", PROJECT_ROOT.exists())
print("Data exists:", DATA_DIR.exists())
print("Train exists:", (DATA_DIR / "train_data.csv").exists())

train = pd.read_csv(DATA_DIR / "train_data.csv")
test = pd.read_csv(DATA_DIR / "test_data.csv")
labels = pd.read_csv(DATA_DIR / "train_labels.csv")

print("Train:", train.shape)
print("Test:", test.shape)

Project exists: True
Data exists: True
Train exists: True
Train: (5531451, 190)
Test: (11363762, 190)


In [3]:
print("QUESTION: Can we distinguish train from test?")

# Keep all numeric features, remove ID and date

train_adv = (train.drop(columns=["customer_ID", "S_2"]).select_dtypes(include=np.number).copy())
test_adv = (test.drop(columns=["customer_ID", "S_2"]).select_dtypes(include=np.number).copy())

# 0 = train, 1 = test

train_adv["target"] = 0
test_adv["target"] = 1

df_adversarial = pd.concat([train_adv, test_adv], ignore_index=True)

X = df_adversarial.drop(columns="target")
y = df_adversarial["target"]

X_train_adv, X_val_adv, y_train_adv, y_val_adv = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

pipe_classification = make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=500))
pipe_classification.fit(X_train_adv, y_train_adv)

proba = pipe_classification.predict_proba(X_val_adv)[:, 1]
auc = roc_auc_score(y_val_adv,proba)

print("Adversarial ROC-AUC:", auc)

QUESTION: Can we distinguish train from test?


MemoryError: Unable to allocate 15.7 GiB for an array with shape (185, 11363762) and data type float64